### 数据准备

In [ ]:
import numpy as np
state = np.array(['认真复习', '简单复习', '没有复习'])  # 定义隐藏状态：学生的复习状态
grade = np.array(['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+', 'C', 'C-'])  # 定义观测状态：最终考试成绩
n_state = len(state)  # 隐藏状态数量
m_grade = len(grade)  # 观测状态数量
pi = np.ones(n_state) / n_state  # 初始状态概率，假设三个复习状态一开始出现的概率相同，均为 1/3
t = np.array([  # 状态转移概率矩阵，t[i][j] 表示从隐藏状态 i 转移到隐藏状态 j 的概率
    [0.4, 0.3, 0.3],
    [0.3, 0.4, 0.3],
    [0.3, 0.3, 0.4]])
e = np.zeros([3, 9])  # 初始化发射概率矩阵，3 行表示 3 个隐藏状态，9 列表示 9 种成绩
e[0, :9] = 1 / 9    # 认真复习时，9 种成绩的概率都设为 1/9
e[1, 3:9] = 1 / 6   # 简单复习时，只可能得到 B+ ～ C-，共 6 种成绩
e[2, 5:9] = 1 / 4   # 没有复习时，只可能得到 B- ～ C-，共 4 种成绩

In [ ]:
print('初始概率矩阵：\n', pi)
print('转移矩阵：\n', t)
print('发射矩阵：\n', e)

### hmmlearn

In [ ]:
from hmmlearn.hmm import CategoricalHMM
# 创建离散型隐马尔可夫模型，n_state 表示隐藏状态的数量
hmm = CategoricalHMM(n_state)

In [ ]:
hmm.startprob_ = pi     # 设置初始状态概率
hmm.transmat_ = t       # 设置隐藏状态之间的转移概率矩阵
hmm.emissionprob_ = e   # 设置隐藏状态生成观测值的发射概率矩阵
hmm.n_feature = 9       # 设置观测值类别数量，共 9 种成绩

In [ ]:
# 构造观测序列，数字 0~8 分别对应 A+、A、A-、B+、B、B-、C+、C、C-
datas = np.array([0, 4, 2, 6, 1])
# CategoricalHMM 要求输入为二维数组，将 shape 从 (5,) 转换为 (5, 1)
datas = np.expand_dims(datas, axis=1)
states = hmm.predict(datas)  # 根据观测序列推断最可能的隐藏状态序列
states

In [ ]:
prob = hmm.score(datas)  # 计算观测序列的对数似然值，score() 返回 log P(X)
prob  # 查看对数概率

In [ ]:
print(np.exp(prob))  # 将对数概率转换回普通概率，P(X) = exp(log P(X))

In [ ]:
datas, states = hmm.sample(10000)  # 根据当前HMM随机生成10000个观测值，datas：生成的观测序列，states：对应的隐藏状态序列

In [ ]:
t_2 = np.zeros([3, 3])  # 创建 3×3 的矩阵，用于统计实际生成数据中的状态转移概率
for i in range(3):  # 遍历每一种当前隐藏状态
    current = np.where(states == i)[0]  # 找出隐藏状态 i 出现的位置
    next_index = current + 1  # 获取这些位置的下一个位置
    next_index = next_index[:-1]  # 删除最后一个索引，防止超出 states 的范围
    tmp = states[next_index]  # 获取状态 i 后面实际出现的隐藏状态
    for j in range(3):  # 统计从状态 i 转移到状态 j 的概率
        t_2[i][j] = np.where(tmp == j)[0].shape[0] / np.shape(tmp)[0]
print(t_2)  # 输出根据随机样本估计得到的转移矩阵

In [ ]:
e_2 = np.zeros([3, 9])  # 创建 3×9 的矩阵，用于统计每个隐藏状态对应各观测值的发射概率
for i in range(3):  # 遍历 3 个隐藏状态
    current = np.where(states == i)[0]  # 找出隐藏状态为 i 的所有位置索引
    next_index = current + 1            # 计算下一位置索引
    next_index = next_index[:-1]        # 去掉最后一个索引，防止越界
    tmp = datas[current]                # 取出隐藏状态 i 对应的所有观测值
    for j in range(9):                  # 遍历 9 种可能的观测值
        e_2[i][j] = np.where(tmp == j)[0].shape[0] / np.shape(tmp)[0]# 统计观测值 j 在当前隐藏状态下出现的比例，即估计 P(观测值=j | 隐藏状态=i)
print(e_2)  # 输出根据样本统计得到的发射概率矩阵